<img src="https://huggingface.co/datasets/FineEnvs/SmolDataEnvs/resolve/main/banner.png" width="100%">

# 2 · Reinforcement learning on SmolDataEnvs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adithya-s-k/FineEnvs/blob/main/04-smoldataenvs/notebooks/02_rl.ipynb)

**Needs an A100 runtime** (Colab Pro, or the Hugging Face Jobs command at the end). RL generates with
vLLM on the same GPU it trains on, which does not fit on a T4. The SFT notebook does.
Nothing to install beyond the first cell, nothing to configure to get a first result.

RL is practice: the model writes a program, a sandbox runs it, and a grader pays it for being right.

**You will:** build the environment in about 30 lines, watch it grade a real answer,
and run GRPO on it.

> **New to this?** Run the cells in order. Every cell prints what it did, and the two
> configuration lines you might want to change (`MODEL`, `NUM_TASKS`) are marked where they
> appear. If a cell fails, the most common cause is a missing GPU runtime:
> *Runtime → Change runtime type → A100 GPU*.


In [ ]:
%pip install -q trl transformers datasets huggingface_hub math-verify trackio


## 1. A task

Each row of `SmolDataEnvs` is a question about a real Kaggle table plus the answer a human
computed in a notebook. The table is not in the prompt. The model has to open it.


In [ ]:
from datasets import load_dataset

ds = load_dataset("FineEnvs/SmolDataEnvs", split="train")
row = ds[0]
print("question :", row["question"])
print("answer   :", row["answer"], f"({row['reward_mode']})")
print("files    :", list(row["files"]))


## 2. The sandbox

[Hugging Face Sandboxes](https://huggingface.co/docs/huggingface_hub/main/guides/sandbox) are
throwaway VMs. The image already has pandas and a script that pulls a task's tables into
`/home/user/input`.

One sandbox serves the whole run: starting one costs ~8 seconds, and GRPO asks for several
attempts at the *same* task in a row, so the data is pulled once and reused.

Sandboxes run on your Hugging Face account, so log in first.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from huggingface_hub import Sandbox

sb = Sandbox.create(
    image="docker.io/savatar101/env-data-agent-train:base",
    flavor="cpu-basic", idle_timeout="30m", forward_hf_token=True,
)

# stage this task's tables, exactly as the environment does during training
sb.run(f"rm -rf /home/user/input/* && BUCKET_PREFIX={row['bucket_prefix']} python3 /opt/pull_bucket.py", check=False)
print(sb.run("ls /home/user/input").stdout)


## 3. Running a program in it

`check=False` matters: the model's program failing is the *normal* case in RL. A traceback is a
reward of zero and a gradient, not an error to raise on.


In [ ]:
def run_program(code: str) -> str:
    """Run code against the task's tables; return whatever it printed last."""
    payload = code.replace("'", "'\\''")
    r = sb.run(
        f"cd /home/user/input && printf '%s' '{payload}' > /tmp/s.py && timeout 90 python3 /tmp/s.py",
        check=False,
    )
    lines = [ln.strip() for ln in (r.stdout or "").splitlines() if ln.strip()]
    return lines[-1] if lines else ""


program = """
import pandas as pd, glob
df = pd.read_csv(glob.glob('*.csv')[0], low_memory=False)
print(df.shape)
"""
print("printed:", run_program(program))


## 4. The reward

`grader.py` ships with the dataset, so training and the dataset agree on what *correct* means.


In [ ]:
from huggingface_hub import hf_hub_download
import importlib.util, sys

path = hf_hub_download("FineEnvs/SmolDataEnvs", "grader.py", repo_type="dataset")
spec = importlib.util.spec_from_file_location("grader", path)
grader = importlib.util.module_from_spec(spec)
sys.modules["grader"] = grader          # its dataclasses need this
spec.loader.exec_module(grader)

def reward(prediction: str) -> float:
    if not prediction:
        return 0.0
    r = grader.grade(row["answer"], prediction, reward_mode=row["reward_mode"],
                     abs_tol=row["atol"], rel_tol=row["rtol"])
    return float(r.reward)

print("gold  ->", reward(row["answer"]))
print("wrong ->", reward("123"))


## 5. The whole environment

That is everything: prompt in, reward out. The reward is one number: **1.0** when the printed
value matches the gold answer under the grader, **0.0** otherwise.

There is no partial credit for a program that merely runs. Early on almost nothing is correct, so
it is tempting to add some, but a shaping term like that is easier to earn than the real answer,
and the policy will optimise it instead.


In [ ]:
import re

SYSTEM = "You are a data analyst. You answer questions about CSV files by writing a short Python program and reading what it prints."

def build_prompt(r):
    files = "\n".join(f"- {f}" for f in r["files"])
    user = (
        f"{r['question']}\n\n"
        f"The files are in /home/user/input and your program runs in that directory:\n{files}\n\n"
        "Write one Python program in a ```python block, then stop.\n\n"
        "- Look at the data if you need to, then compute the answer.\n"
        "- The LAST thing the program prints must be the answer on its own.\n"
        "- Keep it under 40 lines."
    )
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def extract_code(completion):
    text = completion[-1]["content"] if isinstance(completion, list) else completion
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", text or "", re.S)
    return (blocks[-1] if blocks else text or "").strip()


## 6. Train

GRPO samples several programs per task, grades them, and pushes the policy toward the ones that
scored higher. Generation runs *inside* the trainer with vLLM (`colocate`), so there is no
separate inference server.

This cell is a demonstration on a handful of tasks. A real run is the launcher below.


In [ ]:
import torch
from trl import GRPOConfig, GRPOTrainer

MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"   # "Qwen/Qwen3.5-2B" for the real thing
NUM_TASKS = 8

# A mix of tiers. GRPO learns from disagreement inside a group, so a tier the
# model never solves contributes nothing -- but train on easy alone and you get a
# model that is good at easy tasks. Narrow it to debug, widen it to believe.
TIERS = {"easy", "medium"}          # {"easy"} while you debug the loop
subset = ds.filter(lambda r: r["difficulty_tier"] in TIERS)
train = subset.select(range(NUM_TASKS)).map(
    lambda r: {"prompt": build_prompt(r)}, load_from_cache_file=False
)
keep = {"prompt", "answer", "reward_mode", "atol", "rtol", "bucket_prefix"}
train = train.remove_columns([c for c in train.column_names if c not in keep])

def reward_correct(completions, **cols):
    out = []
    for i, c in enumerate(completions):
        sb.run(f"rm -rf /home/user/input/* && BUCKET_PREFIX={cols['bucket_prefix'][i]} python3 /opt/pull_bucket.py", check=False)
        printed = run_program(extract_code(c))
        r = grader.grade(cols["answer"][i], printed, reward_mode=cols["reward_mode"][i],
                         abs_tol=cols["atol"][i], rel_tol=cols["rtol"][i]) if printed else None
        out.append(float(r.reward) if r else 0.0)
    return out

trainer = GRPOTrainer(
    model=MODEL,
    reward_funcs=reward_correct,
    train_dataset=train,
    args=GRPOConfig(
        output_dir="smoldataenvs-grpo",
        num_generations=4,
        per_device_train_batch_size=4,
        max_steps=5,
        learning_rate=3e-6,
        temperature=0.8,
        # everything below was learned the hard way -- see the next cell
        max_completion_length=1024,
        mask_truncated_completions=True,
        repetition_penalty=1.05,
        chat_template_kwargs={"enable_thinking": False},
        gradient_checkpointing=True,
        bf16=torch.cuda.is_available(),
        logging_steps=1,
    ),
)
trainer.train()


In [ ]:
sb.kill()   # sandboxes idle out on their own, but be tidy


## 7. The real run, on Hugging Face Jobs

`train_grpo.py` is this notebook as one file, with the config from the run behind the curves on
the dataset card: lr 3e-6, temperature 0.8, 8 generations, an effective batch of 16.

A 2B full fine-tune with colocated vLLM needs an **a100-large**.

Measure first, train, measure again, with the same script on both sides, or the numbers are not
comparable.


In [ ]:
SHA = "main"   # pin a commit for a reproducible run
RAW = f"https://raw.githubusercontent.com/adithya-s-k/FineEnvs/{SHA}/04-smoldataenvs/scripts"

# 1 · baseline
!hf jobs uv run --detach --flavor a10g-large --timeout 2h --image huggingface/trl --secrets HF_TOKEN \
  -e MODEL=Qwen/Qwen3.5-2B -e ROLLOUT_URL={RAW}/rollout.py {RAW}/eval_pass1.py

# 2 · train
!hf jobs uv run --detach --flavor a100-large --timeout 6h --image huggingface/trl --secrets HF_TOKEN \
  -e MODEL=Qwen/Qwen3.5-2B -e HUB_MODEL_ID=your-name/smoldataenvs-grpo-2b \
  -e TRACKIO_SPACE_ID=your-name/trackio -e ROLLOUT_URL={RAW}/rollout.py {RAW}/train_grpo.py

# 3 · measure again
# !hf jobs uv run --detach --flavor a10g-large --timeout 2h --image huggingface/trl --secrets HF_TOKEN \
#   -e MODEL=your-name/smoldataenvs-grpo-2b -e ROLLOUT_URL={RAW}/rollout.py {RAW}/eval_pass1.py


Run the eval once **before** training and once after, on the whole 144-task split. A
quick subset is not large enough to separate a real change from noise, and a short run
produces changes smaller than that noise either way.

· [the dataset](https://huggingface.co/collections/FineEnvs/smoldataenvs)
· [the code](https://github.com/adithya-s-k/FineEnvs/tree/main/04-smoldataenvs)
· [01 · SFT](https://colab.research.google.com/github/adithya-s-k/FineEnvs/blob/main/04-smoldataenvs/notebooks/01_sft.ipynb)
